# 16 — HTTP, JSON APIs, and Reliable Clients

HTTP clients request remote resources and translate them into application data. Learn methods, status handling, timeouts, JSON parsing, and testability. All runnable code uses local stubs—no network access or API tokens are required.

*Intermediate · 75–120 min*

## You will learn

- Construct requests and reason about GET and POST
- Decode JSON responses and validate schemas
- Set timeouts and handle HTTP/URL failures
- Mock remote calls and reason about pagination and retries

## Examples

### 1. HTTP request model

GET normally retrieves data and should not change server state. POST generally submits data and may have side effects. Headers describe content and accepted formats; a timeout avoids waiting indefinitely. Never embed secret tokens in public notebooks.

In [ ]:
from urllib.request import Request

request = Request(
    "https://example.invalid/api/items?page=1",
    headers={"Accept": "application/json", "User-Agent": "PythonCourse/1.0"},
    method="GET",
)
print(request.get_method())
print(request.full_url)
print(request.get_header("Accept"))

**Think about it:** Why is a GET request not automatically harmless if the remote service is badly designed?

### 2. JSON decoding with a stub

A response body is bytes that must be decoded according to its encoding. Then JSON parsing yields Python values. The data may be a list or dictionary; always validate the expected schema. Here a BytesIO object acts as a fake response.

In [ ]:
import json
from io import BytesIO

body = BytesIO(b'{"items": [{"id": 1, "name": "Pen"}], "next": null}')
payload = json.loads(body.read().decode("utf-8"))
if not isinstance(payload, dict) or not isinstance(payload.get("items"), list):
    raise ValueError("unexpected response shape")
print(payload["items"])
print(payload["next"])

**Think about it:** What would happen if the response contained HTML instead of JSON?

### 3. Mock urlopen rather than using the network

By injecting a function that opens URLs, callers can use a mock during testing. This isolates HTTP behavior from data interpretation and prevents flaky tests that depend on external services.

In [ ]:
import json
from io import BytesIO
from urllib.request import Request

def get_json(url, *, opener, timeout=5):
    request = Request(url, headers={"Accept": "application/json"}, method="GET")
    with opener(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def fake_open(request, timeout):
    assert request.get_method() == "GET"
    assert timeout > 0
    return BytesIO(b'{"status": "ok", "count": 3}')

print(get_json("https://example.invalid/items", opener=fake_open))

**Think about it:** Which errors might a real opener raise that the fake never raises?

### 4. Pagination and controlled retry strategies

Many APIs return pages of results. An explicit next link or page token guides iteration. Clients should cap pages to avoid infinite loops. A retry policy may be useful for transient failures, but blindly retrying POST requests risks duplicate actions.

In [ ]:
PAGES = {
    1: {"items": ["A", "B"], "next": 2},
    2: {"items": ["C"], "next": None},
}
def collect_pages(fetch_page, first_page=1, max_pages=10):
    results, page, visited = [], first_page, set()
    while page is not None:
        if page in visited or len(visited) >= max_pages:
            raise RuntimeError("pagination loop or page limit reached")
        visited.add(page)
        response = fetch_page(page)
        results.extend(response["items"])
        page = response.get("next")
    return results

print(collect_pages(lambda page: PAGES[page]))

**Think about it:** Why is checking repeated page tokens useful?

## Exercises

**Exercise 1.** Write a response validator for a JSON list of products.

In [ ]:
# Exercise 1
# Implement your solution here, and add tests.

**Exercise 2.** Build an API wrapper whose network opener can be replaced by a mock.

In [ ]:
# Exercise 2
# Implement your solution here, and add tests.

**Exercise 3.** Extend pagination to stop after a maximum number of pages.

In [ ]:
# Exercise 3
# Implement your solution here, and add tests.

**Exercise 4.** Describe how you would handle status 429 and a Retry-After header without retrying forever.

In [ ]:
# Exercise 4
# Implement your solution here, and add tests.

## Common mistakes

- Always set network timeouts in real clients.
- Do not make live network requests in deterministic teaching tests.
- Retries need limits and awareness of idempotency; never assume all POSTs can safely repeat.

**Next:** [17 — Command-Line Interfaces and Project Packaging](17_cli_and_packaging.ipynb)